for ($i = 1; $i -le 30; $i++) {
  Get-Content Instances/instanceX.txt | c:/Users/gabri/Documents/Strip-Packing-Verification-tool-main/.venv/Scripts/python.exe Strip_packing_MKBL_v04.py *> $null
}

## Estatísticas Gerais

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
import os
import glob

# Encontrar automaticamente todos os CSVs gerados com padrão instance_*_*.csv
instance_csvs = glob.glob('results/instance_*.csv')

# Organizar por instância e algoritmo
instances_dict = {}
for csv_path in sorted(instance_csvs):
    base = os.path.splitext(os.path.basename(csv_path))[0]  # e.g. instance_three_area
    parts = base.split('_')
    if len(parts) >= 3:
        instance_name = '_'.join(parts[1:-1])  # e.g. three
        algorithm = parts[-1]  # e.g. area, choice, len
        if instance_name not in instances_dict:
            instances_dict[instance_name] = {}
        instances_dict[instance_name][algorithm] = csv_path

# Ordem desejada das instâncias
instance_order = ['three', 'threep2', 'threep3', 'threep2w9', 'threep3w9']

# Carregar e agregar dados
dfs = {}
names = {}
Algoritmos = {}
csv_paths = {}
idx = 0

for instance_name in instance_order:
    if instance_name in instances_dict:
        for algorithm in ['area', 'ratio', 'len']:
            if algorithm in instances_dict[instance_name]:
                csv_path = instances_dict[instance_name][algorithm]
                df = pd.read_csv(csv_path)
                dfs[idx] = df
                names[idx] = instance_name
                Algoritmos[idx] = algorithm
                csv_paths[idx] = csv_path
                idx += 1

print(f"Carregados {len(dfs)} CSVs:")
for i in range(len(dfs)):
    print(f"  {i}: {names.get(i)} - {Algoritmos.get(i)} -> {os.path.basename(csv_paths.get(i))}")

if dfs:
    print(f"\nExemplo (primeiros dados do primeiro CSV):")
    print(dfs[0].head())

Carregados 15 CSVs:
  0: three - area -> instance_three_area.csv
  1: three - ratio -> instance_three_ratio.csv
  2: three - len -> instance_three_len.csv
  3: threep2 - area -> instance_threep2_area.csv
  4: threep2 - ratio -> instance_threep2_ratio.csv
  5: threep2 - len -> instance_threep2_len.csv
  6: threep3 - area -> instance_threep3_area.csv
  7: threep3 - ratio -> instance_threep3_ratio.csv
  8: threep3 - len -> instance_threep3_len.csv
  9: threep2w9 - area -> instance_threep2w9_area.csv
  10: threep2w9 - ratio -> instance_threep2w9_ratio.csv
  11: threep2w9 - len -> instance_threep2w9_len.csv
  12: threep3w9 - area -> instance_threep3w9_area.csv
  13: threep3w9 - ratio -> instance_threep3w9_ratio.csv
  14: threep3w9 - len -> instance_threep3w9_len.csv

Exemplo (primeiros dados do primeiro CSV):
                  Algoritmo  Allocated Pieces  Maximum Length on Strip  \
0  Bottom-Left (Area Order)                 6                      8.0   
1  Bottom-Left (Area Order)         

In [6]:
# Agregar dados de cada CSV por instância e algoritmo
summary_rows = []

# Carregar resumo de quantas rodadas foram necessárias para 1s, se existir
runs_summary = None
if os.path.exists('runs_to_1s_summary.csv'):
    try:
        runs_summary = pd.read_csv('runs_to_1s_summary.csv')
    except Exception:
        runs_summary = None

for idx, df in dfs.items():
    instance_name = names[idx]
    algorithm = Algoritmos[idx]
    csv_path = csv_paths.get(idx)

    # Estatísticas básicas
    mean_time = df['Total Allocation Time (s)'].astype(float).mean() if 'Total Allocation Time (s)' in df.columns else float('nan')
    max_length = df['Maximum Length on Strip'].astype(float).max() if 'Maximum Length on Strip' in df.columns else float('nan')
    rows = len(df)

    # Encontrar rodadas necessárias para completar 1s a partir do resumo gerado pelo script
    runs_needed = None
    if runs_summary is not None and csv_path is not None:
        key = os.path.splitext(os.path.basename(csv_path))[0]  # ex: instance_three_area
        match = runs_summary[runs_summary['instance'] == key]
        if not match.empty:
            try:
                runs_needed = int(match.iloc[-1]['runs_needed'])
            except Exception:
                runs_needed = None

    summary_rows.append({
        'Instancia': instance_name,
        'Algoritmo': algorithm,
        'Rodadas para 1s': runs_needed if runs_needed is not None else '',
        'Tempo Médio (s)': round(mean_time, 6) if not pd.isna(mean_time) else '',
        'Comprimento Máximo': round(max_length, 2) if not pd.isna(max_length) else ''
    })

summary_df = pd.DataFrame(summary_rows)

# Garantir ordem correta das categorias
summary_df['Algoritmo'] = pd.Categorical(summary_df['Algoritmo'], categories=['area', 'ratio', 'len'], ordered=True)
summary_df['Instancia'] = pd.Categorical(
    summary_df['Instancia'],
    categories=['three', 'threep2', 'threep3', 'threep2w9', 'threep3w9'],
    ordered=True
)

# Pivotar: instâncias como linhas, algoritmos como colunas (métricas como subcolunas)
summary_table = summary_df.pivot(index='Instancia', columns='Algoritmo')
summary_table.columns = pd.MultiIndex.from_tuples(
    [(algoritmo, medida) for medida, algoritmo in summary_table.columns],
    names=['Algoritmo', 'Métrica']
)

# Reorganizar para métricas desejadas por algoritmo
metrics = ['Rodadas para 1s', 'Tempo Médio (s)', 'Comprimento Máximo']
summary_table = summary_table.reindex(columns=pd.MultiIndex.from_product([
    ['area', 'ratio', 'len'],
    metrics
], names=['Algoritmo', 'Métrica']))

summary_table.style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]} 
]).format(na_rep='')